# POS Tagging Baselines: XLM-R, mBERT, MuRIL

Baseline models for Hindi POS tagging on the HDTB treebank.
Compare these against the MRL-POS Hindi notebook.

**Models:**
1. **XLM-RoBERTa-base** + Linear + CRF (same encoder as MRL-POS, no affix branch)
2. **mBERT** (multilingual BERT) + Linear + CRF
3. **MuRIL** (Google, trained on 17 Indian languages) + Linear + CRF

All use the same data, tags, training config, and evaluation for fair comparison.

In [ ]:
!pip install -q transformers pytorch-crf conllu scikit-learn

In [ ]:
# Cell 2b: Mount Google Drive for persistent checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/mrl_pos_checkpoints'
    import os
    os.makedirs(CKPT_DIR, exist_ok=True)
    COLAB = True
    print(f"✓ Google Drive mounted. Checkpoints: {CKPT_DIR}")
except:
    COLAB = False
    CKPT_DIR = './'
    print("⚠ Not in Colab. Checkpoints will be saved locally.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from torchcrf import CRF
from sklearn.metrics import f1_score, accuracy_score, classification_report
import os, time
import conllu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def download_hdtb():
    ud_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master"
    splits = {
        "train": "hi_hdtb-ud-train.conllu",
        "dev":   "hi_hdtb-ud-dev.conllu",
        "test":  "hi_hdtb-ud-test.conllu",
    }
    os.makedirs("hdtb", exist_ok=True)
    for split, fname in splits.items():
        path = f"hdtb/{fname}"
        if not os.path.exists(path):
            print(f"Downloading {split}...")
            !wget -q {ud_url}/{fname} -O {path}
    return splits

def load_conllu(path):
    data = []
    with open(path, 'r') as f:
        for sent in conllu.parse(f.read()):
            words = [tok["form"] for tok in sent]
            tags = [tok["upos"] for tok in sent]
            data.append((words, tags))
    return data

splits = download_hdtb()
train_data = load_conllu(f"hdtb/{splits['train']}")
dev_data   = load_conllu(f"hdtb/{splits['dev']}")
test_data  = load_conllu(f"hdtb/{splits['test']}")

# Build tag set from training data
all_tags = set()
for _, tags in train_data:
    all_tags.update(tags)
TAG2IDX = {"PAD": 0}
for i, t in enumerate(sorted(all_tags), 1):
    TAG2IDX[t] = i
IDX2TAG = {v: k for k, v in TAG2IDX.items()}
NUM_TAGS = len(TAG2IDX)

print(f"Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}")
print(f"Tags ({NUM_TAGS}): {TAG2IDX}")

## 1. Load Hindi HDTB Data

In [ ]:
def download_hdtb():
    ud_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master"
    splits = {
        "train": "hi_hdtb-ud-train.conllu",
        "dev":   "hi_hdtb-ud-dev.conllu",
        "test":  "hi_hdtb-ud-test.conllu",
    }
    os.makedirs("hdtb", exist_ok=True)
    for split, fname in splits.items():
        path = f"hdtb/{fname}"
        if not os.path.exists(path):
            print(f"Downloading {split}...")
            !wget -q {ud_url}/{fname} -O {path}
    return splits

def load_conllu(path):
    data = []
    with open(path, 'r') as f:
        for sent in conllu.parse(f.read()):
            words = [tok["form"] for tok in sent]
            tags = [tok["upos"] for tok in sent]
            data.append((words, tags))
    return data

splits = download_hdtb()
train_data = load_conllu(f"hdtb/{splits['train']}")
dev_data   = load_conllu(f"hdtb/{splits['dev']}")
test_data  = load_conllu(f"hdtb/{splits['test']}")

# Build tag set from training data
all_tags = set()
for _, tags in train_data:
    all_tags.update(tags)
TAG2IDX = {"PAD": 0}
for i, t in enumerate(sorted(all_tags), 1):
    TAG2IDX[t] = i
IDX2TAG = {v: k for k, v in TAG2IDX.items()}
NUM_TAGS = len(TAG2IDX)

print(f"Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}")
print(f"Tags ({NUM_TAGS}): {TAG2IDX}")

## 2. Dataset & Model (generic for any HuggingFace encoder)

In [ ]:
def evaluate(model, dataloader):
    """Token-level accuracy and macro F1, excluding PAD."""
    model.eval()
    all_preds, all_golds = [], []
    with torch.no_grad():
        for batch in dataloader:
            preds = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
            )
            for b in range(len(preds)):
                for pred_idx, gold, m in zip(
                    preds[b], batch["tags"][b], batch["attention_mask"][b]
                ):
                    if m.item() == 1 and gold.item() != 0:
                        all_preds.append(pred_idx)
                        all_golds.append(gold.item())

    acc = accuracy_score(all_golds, all_preds)
    macro_f1 = f1_score(all_golds, all_preds, average="macro", zero_division=0)
    return acc, macro_f1, all_preds, all_golds


def train_model(model_name, train_data, dev_data, tag2idx, num_tags,
                max_epochs=30, patience=3, batch_size=32, max_len=128, lr=3e-5):
    """Full train+eval pipeline for one baseline model."""
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds = POSDataset(train_data, tokenizer, tag2idx, max_len)
    dev_ds   = POSDataset(dev_data, tokenizer, tag2idx, max_len)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    dev_dl   = DataLoader(dev_ds, batch_size=batch_size)

    model = BaselinePOSTagger(model_name, num_tags).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    best_f1 = 0
    no_improve = 0
    start = time.time()
    
    # Create safe model name for checkpoint
    safe_name = model_name.replace("/", "_")
    ckpt_path = f"{CKPT_DIR}/{safe_name}_best.pt"
    
    # Try loading checkpoint if it exists
    start_epoch = 0
    if os.path.exists(ckpt_path):
        print(f"Loading checkpoint from {ckpt_path}...")
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt.get('epoch', 0)
        best_f1 = ckpt.get('best_f1', 0)
        print(f"  Resumed from epoch {start_epoch}, Best F1={best_f1:.4f}")

    for epoch in range(start_epoch, max_epochs):
        model.train()
        total_loss = 0
        for batch in train_dl:
            optimizer.zero_grad()
            loss = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                tags=batch["tags"].to(device),
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_dl)
        acc, macro_f1, _, _ = evaluate(model, dev_dl)
        elapsed = time.time() - start
        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | "
              f"Acc: {acc:.4f} | Macro-F1: {macro_f1:.4f} | {elapsed:.0f}s")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_acc = acc
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            # Save checkpoint
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': best_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
            }, ckpt_path)
            print(f"  ✓ Saved checkpoint (F1={best_f1:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    # Final report with best model
    model.load_state_dict(best_state)
    model.to(device)
    acc, macro_f1, all_preds, all_golds = evaluate(model, dev_dl)

    idx2tag = {v: k for k, v in tag2idx.items()}
    pred_labels = [idx2tag[p] for p in all_preds]
    gold_labels = [idx2tag[g] for g in all_golds]

    print(f"\nBest Dev Acc: {acc:.4f} | Best Dev Macro-F1: {macro_f1:.4f}")
    print(classification_report(gold_labels, pred_labels, zero_division=0))

    return {
        "model_name": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "params": total_params,
        "report": classification_report(gold_labels, pred_labels,
                                        zero_division=0, output_dict=True),
    }


print("Training pipeline defined.")

## 3. Training & Evaluation Functions

## 4. Run All Baselines

| Model | What it is | Params | Pre-training languages |
|-------|-----------|--------|------------------------|
| **xlm-roberta-base** | Facebook's multilingual RoBERTa | 278M | 100 languages |
| **bert-base-multilingual-cased** | Google's mBERT | 179M | 104 languages |
| **google/muril-base-cased** | Google's MuRIL (Indian languages) | 237M | 17 Indian + English |

In [ ]:
# Run baselines one by one (each takes ~10-20 min on T4)

BASELINES = [
    "xlm-roberta-base",
    "bert-base-multilingual-cased",
    "google/muril-base-cased",       # publicly available, no gating
]

results = []
for model_name in BASELINES:
    try:
        r = train_model(
            model_name, train_data, dev_data, TAG2IDX, NUM_TAGS,
            max_epochs=30, patience=3, batch_size=32, max_len=128, lr=3e-5
        )
        results.append(r)
    except Exception as e:
        print(f"\nFailed: {model_name} — {e}")
        results.append({"model_name": model_name, "accuracy": 0, "macro_f1": 0,
                        "params": 0, "error": str(e)})

## 5. Comparison Table

In [ ]:
# Comparison table including MRL-POS result

print("\n" + "=" * 75)
print("HINDI POS TAGGING — BASELINE COMPARISON (Dev Set)")
print("=" * 75)
print(f"{'Model':<35} {'Params':>10} {'Acc':>8} {'Macro-F1':>10}")
print("-" * 75)

for r in results:
    name = r['model_name'].split('/')[-1]
    params = f"{r['params']/1e6:.1f}M" if r['params'] else "N/A"
    acc = f"{r['accuracy']:.4f}" if r['accuracy'] else "FAILED"
    f1 = f"{r['macro_f1']:.4f}" if r['macro_f1'] else "FAILED"
    print(f"{name:<35} {params:>10} {acc:>8} {f1:>10}")

# Add MRL-POS reference line
print("-" * 75)
print(f"{'MRL-POS Hindi (your model)':<35} {'~279M':>10} {'0.9784':>8} {'0.9349':>10}")
print("=" * 75)
print("\nNote: MRL-POS has ~1M extra params (affix module + co-attention)")
print("on top of xlm-roberta-base.")

## 6. Per-Tag Comparison Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# MRL-POS results from your run
mrl_pos_f1 = {
    "ADJ": 0.95, "ADP": 1.00, "ADV": 0.89, "AUX": 0.99,
    "CCONJ": 1.00, "DET": 0.97, "NOUN": 0.97, "NUM": 0.98,
    "PART": 0.99, "PRON": 0.99, "PROPN": 0.95, "PUNCT": 1.00,
    "SCONJ": 1.00, "VERB": 0.99,
}

# Tags to compare (skip X — too few samples)
compare_tags = sorted(mrl_pos_f1.keys())

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(compare_tags))
width = 0.2

# Plot MRL-POS
mrl_vals = [mrl_pos_f1[t] for t in compare_tags]
ax.bar(x - 1.5*width, mrl_vals, width, label="MRL-POS Hindi", color="#2ecc71")

# Plot each baseline
colors = ["#3498db", "#e74c3c", "#9b59b6"]
for i, r in enumerate(results):
    if "report" in r:
        vals = [r["report"].get(t, {}).get("f1-score", 0) for t in compare_tags]
        name = r["model_name"].split("/")[-1]
        ax.bar(x + (i-0.5)*width, vals, width, label=name, color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels(compare_tags, rotation=45, ha="right", fontsize=10)
ax.set_ylabel("F1 Score")
ax.set_title("Per-Tag F1: MRL-POS vs Baselines (Hindi HDTB Dev)")
ax.legend(loc="lower left")
ax.set_ylim(0.7, 1.02)
ax.axhline(y=0.95, color="gray", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. What to Look For

When comparing MRL-POS against these baselines, focus on:

1. **ADJ, ADV, PROPN** — These are the ambiguous tags where affix info should help most
2. **VERB** — Hindi verb suffixes (-ता/-ती/-ते) are strong POS signals; MRL-POS should shine here
3. **Overall macro-F1** — MRL-POS should beat all baselines, especially on rare tags
4. **MuRIL** — Interesting because it's specifically trained on Indian languages (237M params)

Expected ranking:
```
MRL-POS Hindi  >  MuRIL  ≥  XLM-R baseline  >  mBERT
```